In [1]:
import psycopg2
import pandas as pd
import warnings

STATUS_TYPE_ID = 5
MODEL_TYPE_ID = 6
NON_TESTED_MODEL_TYPES = [8, 9, 12]
FILTER_LAYERS = True
DB_ARGS = {
    "dbname": "mnpz_oil_quality", 
    "host": "10.49.146.78",
    "user": "nk_python", 
    "password": "guylian"
}


if FILTER_LAYERS:
    LAYER1_FILTER = "%_L1_%"
    LAYER2_FILTER = "%_L2_%"
    LAYER3_FILTER = "%_L3_%"
else:
    LAYER1_FILTER = ""
    LAYER2_FILTER = ""
    LAYER3_FILTER = ""

warnings.filterwarnings('ignore')
pd.set_option('display.max_rows', 2000)
pd.set_option('display.expand_frame_repr', False)

In [3]:
# Поиск моделей без тега статуса
print("="*100)

with psycopg2.connect(**DB_ARGS) as conn:
    query = """
        WITH  models AS (
        	SELECT id, name
        	FROM dictionary.union_tags
        	WHERE source_type_id = %(model_type_id)s
            AND disabled = False
        	AND NOT ((attributes->>'type_model')::INTEGER = ANY(%(non_tested_model_types)s))
            AND name NOT LIKE %(layer1_filter)s
            AND name NOT LIKE %(layer2_filter)s
            AND name NOT LIKE %(layer3_filter)s
        ),
        statuses AS (
        	SELECT id, name, parent_id
        	FROM dictionary.union_tags
        	WHERE source_type_id =  %(status_type_id)s
        ),
        aggr AS (
        	SELECT models.id, models.name, statuses.id AS status_id
        	FROM models
        	LEFT JOIN statuses ON models.id = statuses.parent_id
        )
        SELECT * FROM aggr 
        WHERE status_id IS NULL
    """
    params = {
        "status_type_id": STATUS_TYPE_ID, 
        "model_type_id": MODEL_TYPE_ID, 
        "layer1_filter": LAYER1_FILTER,
        "layer2_filter": LAYER2_FILTER,
        "layer3_filter": LAYER3_FILTER,
        "non_tested_model_types": NON_TESTED_MODEL_TYPES
    }
    no_status_tag_df = pd.read_sql(query, conn, params=params, index_col="id").sort_index()
    if no_status_tag_df.empty:
        print("У всех моделей есть тег статуса")
    else:
        print("У следующих моделей нет тега статуса:")
        print(no_status_tag_df)
        
print("="*100)

У следующих моделей нет тега статуса:
                           name status_id
id                                       
9544        MATOB:P542_CFPP_old      None
23907          MATOB:to_DT_I350      None
23908  MATOB:to_Kerosin_Ssulfur      None
25176              MixEthyl:MON      None
25177              MixBonus:MON      None
25178             MixDupont:MON      None
26051  MATOB:P534_T95_Riazi_old      None
26056  MATOB:P542_T95_Riazi_old      None
26066  MATOB:P544_T95_Riazi_old      None
26071  MATOB:P545_T95_Riazi_old      None
26076  MATOB:P547_T95_Riazi_old      None


In [4]:
# Поиск моделей без значений статуса в таблице union_values
print("="*100)

with psycopg2.connect(**DB_ARGS) as conn:
    query = """
        WITH models AS (
        	SELECT id, name
        	FROM dictionary.union_tags
        	WHERE source_type_id = %(model_type_id)s
			AND NOT ((attributes->>'type_model')::INTEGER = ANY(%(non_tested_model_types)s))
			AND disabled = False
        ),
        statuses AS (
        	SELECT id, name, parent_id
        	FROM dictionary.union_tags
        	WHERE source_type_id =  %(status_type_id)s
        ),
        aggr AS (
        	SELECT models.id, models.name, statuses.id AS status_id
        	FROM models
        	JOIN statuses ON models.id = statuses.parent_id
        ),
        status_values AS (
            SELECT aggr.id, 
    			aggr.name, 
    			aggr.status_id, 
    			MAX(uv.date_value) AS status_date 
    		FROM aggr
    		LEFT JOIN result.union_values uv ON aggr.status_id = uv.tag_id
                AND uv.source_type_id =  %(status_type_id)s
    		GROUP BY aggr.id, aggr.name, aggr.status_id
        )
        SELECT * FROM status_values WHERE status_date IS NULL
    """
    params = {
        "status_type_id": STATUS_TYPE_ID, 
        "model_type_id": MODEL_TYPE_ID,
        "non_tested_model_types": NON_TESTED_MODEL_TYPES
    }
    uv_status_df = pd.read_sql(query, conn, params=params, index_col="id").sort_index()
    if uv_status_df.empty:
        print("У всех моделей есть значения статуса в таблице union_values")
    else:
        print("У следующих моделей нет ни одного значения статуса в таблице union_values:")
        print(uv_status_df)

print("="*100)

У следующих моделей нет ни одного значения статуса в таблице union_values:
                                       name  status_id status_date
id                                                                
2558                MATOB:P518_ChillingTemp      16305        None
2560                MATOB:P522_ChillingTemp      16307        None
2562                MATOB:P524_ChillingTemp      16309        None
2563                  MATOB:P516_SourSulfur      16328        None
2564                  MATOB:P518_SourSulfur      16329        None
2565                  MATOB:P521_SourSulfur      16330        None
2566                  MATOB:P522_SourSulfur      16331        None
2567                  MATOB:P523_SourSulfur      16332        None
2568                  MATOB:P524_SourSulfur      16333        None
2569                        MATOB:to_DT_D15      16352        None
2570                        MATOB:to_DT_MDS      16353        None
2571                 MATOB:to_DT_FlashPoint      16354

In [5]:
# Поиск моделей без значений статуса в таблице union_last_values
print("="*100)

with psycopg2.connect(**DB_ARGS) as conn:
    query = """
        WITH models AS (
        	SELECT id, name
        	FROM dictionary.union_tags
        	WHERE source_type_id = %(model_type_id)s
			AND NOT ((attributes->>'type_model')::INTEGER = ANY(%(non_tested_model_types)s))
			AND disabled = False
        ),
        statuses AS (
        	SELECT id, name, parent_id
        	FROM dictionary.union_tags
        	WHERE source_type_id =  %(status_type_id)s
        ),
        aggr AS (
        	SELECT models.id, models.name, statuses.id AS status_id
        	FROM models
        	JOIN statuses ON models.id = statuses.parent_id
        ),
        status_last_values AS (
            SELECT aggr.id, 
    			aggr.name, 
    			aggr.status_id, 
    			ulv.date_value AS status_date 
    		FROM aggr
    		LEFT JOIN result.union_last_values ulv ON aggr.status_id = ulv.tag_id
                AND ulv.source_type_id =  %(status_type_id)s
        )
        SELECT * FROM status_last_values WHERE status_date IS NULL
    """
    params = {
        "status_type_id": STATUS_TYPE_ID, 
        "model_type_id": MODEL_TYPE_ID,
        "non_tested_model_types": NON_TESTED_MODEL_TYPES
    }
    uv_status_df = pd.read_sql(query, conn, params=params, index_col="id").sort_index()
    if uv_status_df.empty:
        print("У всех моделей есть значение статуса в таблице union_last_values")
    else:
        print("У следующих моделей нет значения статуса в таблице union_last_values:")
        print(uv_status_df)

print("="*100)

У следующих моделей нет значения статуса в таблице union_last_values:
                                     name  status_id status_date
id                                                              
2558              MATOB:P518_ChillingTemp      16305        None
2560              MATOB:P522_ChillingTemp      16307        None
2562              MATOB:P524_ChillingTemp      16309        None
2563                MATOB:P516_SourSulfur      16328        None
2564                MATOB:P518_SourSulfur      16329        None
2565                MATOB:P521_SourSulfur      16330        None
2566                MATOB:P522_SourSulfur      16331        None
2567                MATOB:P523_SourSulfur      16332        None
2568                MATOB:P524_SourSulfur      16333        None
2572               MATOB:to_DT_CloudPoint      16355        None
2574                 MATOB:to_Kerosin_D20      16357        None
2576        MATOB:to_Kerosin_ChillingTemp      17066        None
2577          MATOB:

In [6]:
# Поиск моделей без указанного поля 'lab_mse'
print("="*100)

with psycopg2.connect(**DB_ARGS) as conn:
    query = """
        SELECT id, name
        FROM dictionary.union_tags
        WHERE source_type_id = %(model_type_id)s
        AND NOT ((attributes->>'type_model')::INTEGER = ANY(%(non_tested_model_types)s))
        AND disabled = False
        AND (attributes->>'lab_mse')::REAL IS NULL
        AND name NOT LIKE %(layer1_filter)s
        AND name NOT LIKE %(layer2_filter)s
        AND name NOT LIKE %(layer3_filter)s
    """
    params = {
        "model_type_id": MODEL_TYPE_ID,
        "non_tested_model_types": NON_TESTED_MODEL_TYPES,
        "layer1_filter": LAYER1_FILTER,
        "layer2_filter": LAYER2_FILTER,
        "layer3_filter": LAYER3_FILTER
    }
    no_lab_mse_df = pd.read_sql(query, conn, params=params, index_col="id").sort_index()
    if no_lab_mse_df.empty:
        print("У всех моделей указано поле lab_mse")
    else:
        print("У следующих моделей не указано поле lab_mse:")
        print(no_lab_mse_df)

print("="*100)

У следующих моделей не указано поле lab_mse:
                              name
id                                
2558       MATOB:P518_ChillingTemp
2560       MATOB:P522_ChillingTemp
2562       MATOB:P524_ChillingTemp
2563         MATOB:P516_SourSulfur
2564         MATOB:P518_SourSulfur
2565         MATOB:P521_SourSulfur
2566         MATOB:P522_SourSulfur
2567         MATOB:P523_SourSulfur
2568         MATOB:P524_SourSulfur
2577   MATOB:to_Kerosin_SourSulfur
25982           Pipeline:GODT:CFPP
26002      Pipeline:GODT:T95_Riazi
26006      MATOB:to_P508_T95_Riazi
26008      MATOB:to_P509_T95_Riazi
26010      MATOB:to_P510_T95_Riazi
26012  MATOB:to_P534_T95_Riazi_old
26014  MATOB:to_P542_T95_Riazi_old
26016      MATOB:to_P543_T95_Riazi
26018  MATOB:to_P544_T95_Riazi_old
26020  MATOB:to_P545_T95_Riazi_old
26022  MATOB:to_P547_T95_Riazi_old
27198           MATOB:to_P547_CFPP
27200           MATOB:to_P542_CFPP
27202           MATOB:to_P544_CFPP
27204           MATOB:to_P545_CFPP
27208     

In [7]:
# Поиск моделей без указанного поля 'view_diag'
print("="*100)

with psycopg2.connect(**DB_ARGS) as conn:
    query = """
        SELECT id, name
        FROM dictionary.union_tags
        WHERE source_type_id = %(model_type_id)s
        AND NOT ((attributes->>'type_model')::INTEGER = ANY(%(non_tested_model_types)s))
        AND disabled = False
        AND (attributes->>'view_diag')::BOOLEAN IS NULL
        AND name NOT LIKE %(layer1_filter)s
        AND name NOT LIKE %(layer2_filter)s
        AND name NOT LIKE %(layer3_filter)s
    """
    params = {
        "model_type_id": MODEL_TYPE_ID,
        "non_tested_model_types": NON_TESTED_MODEL_TYPES,
        "layer1_filter": LAYER1_FILTER,
        "layer2_filter": LAYER2_FILTER,
        "layer3_filter": LAYER3_FILTER
    }
    no_lab_mse_df = pd.read_sql(query, conn, params=params, index_col="id").sort_index()
    if no_lab_mse_df.empty:
        print("У всех моделей указано поле view_diag")
    else:
        print("У следующих моделей не указано поле view_diag:")
        print(no_lab_mse_df)

print("="*100)

У всех моделей указано поле view_diag


In [8]:
# Поиск моделей без указанных полей конфигурации ИПК'
print("="*100)

with psycopg2.connect(**DB_ARGS) as conn:
    query = """
        SELECT id, 
            name,
            ((attributes->'ipk')->>'W_r')::REAL AS w_r,
            ((attributes->'ipk')->>'W_r2')::REAL AS w_r2,
            ((attributes->'ipk')->>'W_mae')::REAL AS w_mae,
            ((attributes->'ipk')->>'ipk_window')::INTEGER AS ipk_window
        FROM dictionary.union_tags
        WHERE source_type_id = %(model_type_id)s
        AND NOT ((attributes->>'type_model')::INTEGER = ANY(%(non_tested_model_types)s))
        AND (attributes->>'view_diag')::BOOLEAN=TRUE
        AND disabled = False
        AND (
            ((attributes->'ipk')->>'W_r')::REAL IS NULL
            OR ((attributes->'ipk')->>'W_r2')::REAL IS NULL
            OR ((attributes->'ipk')->>'W_mae')::REAL IS NULL
            OR ((attributes->'ipk')->>'ipk_window')::INTEGER IS NULL
        )
        AND name NOT LIKE %(layer1_filter)s
        AND name NOT LIKE %(layer2_filter)s
        AND name NOT LIKE %(layer3_filter)s
    """
    params = {
        "model_type_id": MODEL_TYPE_ID,
        "non_tested_model_types": NON_TESTED_MODEL_TYPES,
        "layer1_filter": LAYER1_FILTER,
        "layer2_filter": LAYER2_FILTER,
        "layer3_filter": LAYER3_FILTER
    }
    no_ipk_config_df = pd.read_sql(query, conn, params=params, index_col="id").sort_index()
    if no_ipk_config_df.empty:
        print("У всех моделей указана конфигурация ИПК")
    else:
        print("У следующих моделей не указана конфигурация ИПК:")
        print(no_ipk_config_df)

print("="*100)

У всех моделей указана конфигурация ИПК


In [9]:
# Поиск моделей без указанных полей конфигурации дообучения'
print("="*100)

with psycopg2.connect(**DB_ARGS) as conn:
    query = """
        SELECT id, 
            name,
            ((attributes->'train_config')->>'ol_mse_mult')::REAL AS ol_mse_mult,
            ((attributes->'train_config')->>'ol_bias_mult')::REAL AS ol_bias_mult,
            ((attributes->'train_config')->>'ol_window')::INTEGER AS ol_window,
            ((attributes->'train_config')->>'cs_mult')::REAL AS cs_mult,
            ((attributes->'train_config')->>'cs_window')::INTEGER AS cs_window
        FROM dictionary.union_tags
        WHERE source_type_id = %(model_type_id)s
        AND NOT ((attributes->>'type_model')::INTEGER = ANY(%(non_tested_model_types)s))
        AND disabled = False
        AND (attributes->>'view_diag')::BOOLEAN=TRUE
        AND parent_id IS NOT NULL
        AND (
            ((attributes->'train_config')->>'ol_mse_mult')::REAL IS NULL
            OR ((attributes->'train_config')->>'ol_bias_mult')::REAL IS NULL
            OR ((attributes->'train_config')->>'ol_window')::INTEGER IS NULL
            OR ((attributes->'train_config')->>'cs_mult')::REAL IS NULL
            OR ((attributes->'train_config')->>'cs_window')::INTEGER IS NULL
        )
        AND name NOT LIKE %(layer1_filter)s
        AND name NOT LIKE %(layer2_filter)s
        AND name NOT LIKE %(layer3_filter)s
    """
    params = {
        "model_type_id": MODEL_TYPE_ID,
        "non_tested_model_types": NON_TESTED_MODEL_TYPES,
        "layer1_filter": LAYER1_FILTER,
        "layer2_filter": LAYER2_FILTER,
        "layer3_filter": LAYER3_FILTER
    }
    no_train_config_df = pd.read_sql(query, conn, params=params, index_col="id").sort_index()
    if no_train_config_df.empty:
        print("У всех моделей указана конфигурация ИПК")
    else:
        print("У следующих моделей не указана конфигурация ИПК:")
        print(no_train_config_df)

print("="*100)

У всех моделей указана конфигурация ИПК


In [10]:
# Поиск моделей без значений за последний час'
print("="*100)

with psycopg2.connect(**DB_ARGS) as conn:
    query = """
        SELECT ut.id, 
            ut.name,
            ulv.date_value AS last_value_date
        FROM dictionary.union_tags ut
        JOIN result.union_last_values ulv ON ut.id = ulv.tag_id
        WHERE ut.source_type_id = %(model_type_id)s
        AND (attributes->>'view_diag')::BOOLEAN=TRUE
        AND disabled = False
        AND NOT ((ut.attributes->>'type_model')::INTEGER = ANY(%(non_tested_model_types)s))
        AND ulv.date_value < NOW() - interval '1 hour'
    """
    params = {
        "model_type_id": MODEL_TYPE_ID,
        "non_tested_model_types": NON_TESTED_MODEL_TYPES
    }
    ulv_hour_old_values = pd.read_sql(query, conn, params=params, index_col="id").sort_index()
    if ulv_hour_old_values.empty:
        print("У всех моделей есть значения в течение последнего часа")
    else:
        print("У следующих моделей нет значений за последний час:")
        print(ulv_hour_old_values)

print("="*100)

У следующих моделей нет значений за последний час:
                          name           last_value_date
id                                                      
10651                P-577:Tkk 2024-01-31 22:04:43+00:00
10654         P580-600-601:Tkk 2026-01-09 22:12:00+00:00
23971      GFU2:Propane:DNP+45 2026-01-13 08:33:00+00:00
24030             BBF_GRS1:D15 2026-01-18 23:36:00+00:00
24032          BBF_GRS1:DNP+45 2026-01-18 23:36:00+00:00
24034           BBF_GRS1:SumC3 2026-01-18 23:36:00+00:00
24036           BBF_GRS1:SumC4 2026-01-18 23:36:00+00:00
24038           BBF_GRS1:SumC5 2026-01-18 23:36:00+00:00
24040  BBF_GRS1:SumUnsaturated 2026-01-18 23:36:00+00:00


In [11]:
# Поиск отключенных моделей через поле disabled'
print("="*100)

with psycopg2.connect(**DB_ARGS) as conn:
    query = """
        SELECT id, 
            name,
            disabled
        FROM dictionary.union_tags ut
        WHERE ut.source_type_id = %(model_type_id)s
        AND disabled != False
        AND (attributes->>'view_diag')::BOOLEAN=TRUE
        AND NOT ((ut.attributes->>'type_model')::INTEGER = ANY(%(non_tested_model_types)s))
    """
    params = {
        "model_type_id": MODEL_TYPE_ID,
        "non_tested_model_types": NON_TESTED_MODEL_TYPES
    }
    disabled_models = pd.read_sql(query, conn, params=params, index_col="id").sort_index()
    if disabled_models.empty:
        print("Все модели в работе")
    else:
        print("Следующие модели отключены через поле disabled:")
        print(disabled_models)

print("="*100)

Следующие модели отключены через поле disabled:
               name  disabled
id                           
10837          test      True
10838         test1      True
10839  test_cluster      True


In [12]:
# Поиск моделей без привязки к ЛА'
print("="*100)

with psycopg2.connect(**DB_ARGS) as conn:
    query = """
        SELECT id, 
            name,
            parent_id
        FROM dictionary.union_tags ut
        WHERE ut.source_type_id = %(model_type_id)s
        AND parent_id IS NULL
        AND NOT ((ut.attributes->>'type_model')::INTEGER = ANY(%(non_tested_model_types)s))
        AND name NOT LIKE %(layer1_filter)s
        AND name NOT LIKE %(layer2_filter)s
        AND name NOT LIKE %(layer3_filter)s
        AND (attributes->>'view_diag')::BOOLEAN=TRUE
    """
    params = {
        "model_type_id": MODEL_TYPE_ID,
        "non_tested_model_types": NON_TESTED_MODEL_TYPES,
        "layer1_filter": LAYER1_FILTER,
        "layer2_filter": LAYER2_FILTER,
        "layer3_filter": LAYER3_FILTER
    }
    no_parent_models = pd.read_sql(query, conn, params=params, index_col="id").sort_index()
    if no_parent_models.empty:
        print("У всех моделей привязаны ЛА")
    else:
        print("У следующих моделей нет привязки к ЛА:")
        print(no_parent_models)

print("="*100)

У следующих моделей нет привязки к ЛА:
                          name parent_id
id                                      
2686                FIR361:T50      None
3184         G-43-107:DT_Tkk_n      None
3185         G-43-107:DT_T95_n      None
3186         G-43-107:DT_T90_n      None
3187         G-43-107:DT_T50_n      None
3188         G-43-107:DT_IBP_n      None
3189         G-43-107:DT_D15_n      None
10650                P-577:T50      None
10651                P-577:Tkk      None
10652                P-577:Tnk      None
10653         P580-600-601:T50      None
10654         P580-600-601:Tkk      None
10655         P580-600-601:Tnk      None
10825            TAME:315_I100      None
10826            TAME:316_I100      None
10837                     test      None
24030             BBF_GRS1:D15      None
24032          BBF_GRS1:DNP+45      None
24034           BBF_GRS1:SumC3      None
24036           BBF_GRS1:SumC4      None
24038           BBF_GRS1:SumC5      None
24040  BBF_GRS1:Su

In [13]:
# Поиск моделей, у которых не совпадают единицы измерения с единицами измерения ЛА'
print("="*100)

with psycopg2.connect(**DB_ARGS) as conn:
    query = """
        SELECT ut.id, 
            ut.name,
            ut.parent_id,
            ut.unit_id AS unit_id,
            parent_ut.unit_id AS parent_unit_id
        FROM dictionary.union_tags ut
        JOIN dictionary.union_tags parent_ut ON ut.parent_id = parent_ut.id
        WHERE ut.source_type_id = %(model_type_id)s
        AND (ut.attributes->>'view_diag')::BOOLEAN=TRUE
        AND ut.unit_id != parent_ut.unit_id
        AND NOT ((ut.attributes->>'type_model')::INTEGER = ANY(%(non_tested_model_types)s))
        AND ut.name NOT LIKE %(layer1_filter)s
        AND ut.name NOT LIKE %(layer2_filter)s
        AND ut.name NOT LIKE %(layer3_filter)s
        
    """
    params = {
        "model_type_id": MODEL_TYPE_ID,
        "non_tested_model_types": NON_TESTED_MODEL_TYPES,
        "layer1_filter": LAYER1_FILTER,
        "layer2_filter": LAYER2_FILTER,
        "layer3_filter": LAYER3_FILTER
    }
    unit_compare_df = pd.read_sql(query, conn, params=params, index_col="id").sort_index()
    if unit_compare_df.empty:
        print("У всех моделей единицы измерения совпадают с единицами измерения их ЛА")
    else:
        print("У следующих моделей не совпадают единицы измерения с единицами измерения их ЛА:")
        print(unit_compare_df)

print("="*100)

У всех моделей единицы измерения совпадают с единицами измерения их ЛА


In [14]:
# Поиск моделей без указанных пределов tech_min и tech_max'
print("="*100)

with psycopg2.connect(**DB_ARGS) as conn:
    query = """
        SELECT id, 
            name,
            (attributes->>'tech_min')::REAL AS tech_min,
            (attributes->>'tech_max')::REAL AS tech_max
        FROM dictionary.union_tags
        WHERE source_type_id = %(model_type_id)s
        AND (
            (attributes->>'tech_min')::REAL IS NULL
            AND (attributes->>'view_diag')::BOOLEAN=TRUE
            OR (attributes->>'tech_max')::REAL IS NULL
        )
        AND NOT ((attributes->>'type_model')::INTEGER = ANY(%(non_tested_model_types)s))
        AND disabled = False
    """
    params = {
        "model_type_id": MODEL_TYPE_ID,
        "non_tested_model_types": NON_TESTED_MODEL_TYPES
    }
    no_limits_df = pd.read_sql(query, conn, params=params, index_col="id").sort_index()
    if no_limits_df.empty:
        print("У всех моделей указаны пределы tech_min и tech_max")
    else:
        print("У следующих моделей не указаны пределы tech_min и tech_max:")
        print(no_limits_df)

print("="*100)

У следующих моделей не указаны пределы tech_min и tech_max:
                                       name  tech_min tech_max
id                                                            
556                KUPN:U400:N-Butane:SumC5       0.0     None
587                   MTBE:BBF:Butene-2-Cis       0.0     None
613                    KUPN:U250:80-180:IBP       0.0     None
615                        AVT6:NK-85:SumC6       NaN     None
843                       GFU2:Benz:Pentane       0.0     None
973                     GFU2:Benz:Isobutane       0.0     None
981                    MTBE:BBF:H2S+SSulfur       0.0     None
994                       KUPN:U200:RIF:T10       0.0     None
1184                      KUPN:U200:Gaz:D20       0.0     None
1194                 KUPN:U400:N-Butane:MON      80.0     None
1195                 KUPN:U400:N-Butane:RON      90.0     None
1196                     KUPN:U250:FEED:D15       NaN     None
1197                     KUPN:U250:FEED:IBP       NaN     

In [15]:
# Поиск моделей, у которых последнее значение упирается в пределы tech_min или tech_max'
print("="*100)

with psycopg2.connect(**DB_ARGS) as conn:
    query = """
        WITH models AS (
            SELECT ut.id, 
                ut.name,
                (ut.attributes->>'tech_min')::REAL AS tech_min,
                (ut.attributes->>'tech_max')::REAL AS tech_max,
                ulv.real_value AS last_value
            FROM dictionary.union_tags ut
            JOIN result.union_last_values ulv ON ut.id = ulv.tag_id
                AND ut.source_type_id = %(model_type_id)s
            WHERE (
                (attributes->>'tech_min')::REAL IS NOT NULL
                OR (attributes->>'tech_max')::REAL IS NOT NULL
            )
            AND NOT ((attributes->>'type_model')::INTEGER = ANY(%(non_tested_model_types)s))
            AND (attributes->>'view_diag')::BOOLEAN=TRUE
            AND disabled = False
        )
        SELECT *, 
            (last_value <= tech_min)::BOOLEAN AS min_bound,
            (last_value >= tech_max)::BOOLEAN AS max_bound 
        FROM models
        WHERE last_value <= tech_min OR last_value >= tech_max
    """
    params = {
        "model_type_id": MODEL_TYPE_ID,
        "non_tested_model_types": NON_TESTED_MODEL_TYPES
    }
    limited_values_df = pd.read_sql(query, conn, params=params, index_col="id").sort_index()
    limited_values_df["min_bound"] = limited_values_df["min_bound"].fillna(False)
    limited_values_df["max_bound"] = limited_values_df["max_bound"].fillna(False)
    if limited_values_df.empty:
        print("У всех моделей значения не упираются в пределы")
    else:
        print("У следующих моделей значения упираются в пределы tech_min и tech_max:")
        print(limited_values_df)

print("="*100)

У следующих моделей значения упираются в пределы tech_min и tech_max:
                                     name  tech_min  tech_max  last_value  min_bound  max_bound
id                                                                                             
24010              KUPN:U400:Butane:SumC4      75.0     100.0        75.0       True      False
24030                        BBF_GRS1:D15       0.4       0.9         0.9      False       True
24038                      BBF_GRS1:SumC5       0.0     100.0         0.0       True      False
24042                      iBF_GRS3:SumC4       0.0     100.0       100.0      False       True
24052                      BF_GRS4:DNP+45       0.5       2.0         0.5       True      False
24054                       BF_GRS4:SumC4       0.0     100.0       100.0      False       True
24056                       BF_GRS4:SumC5       0.0     100.0         0.0       True      False
24058              BF_GRS4:SumUnsaturated       0.0     100.0     

In [16]:
# Поиск моделей, у которых за изменяются значения за последний час'
print("="*100)

with psycopg2.connect(**DB_ARGS) as conn:
    query = """
        WITH distinct_values AS (
        	SELECT DISTINCT ut.id, ut.name, uv.real_value 
        	FROM dictionary.union_tags ut
        	JOIN result.union_values uv ON ut.id = uv.tag_id
        		AND ut.source_type_id = %(model_type_id)s
        	WHERE uv.date_value >= NOW() - interval '1 hour'
            AND NOT ((ut.attributes->>'type_model')::INTEGER = ANY(%(non_tested_model_types)s))
            AND (attributes->>'view_diag')::BOOLEAN=TRUE
            AND disabled = False
        ),
        count_values AS (
        	SELECT id, name, count(real_value) AS unique_values_count
        	FROM distinct_values
        	GROUP BY id, name
        )
        SELECT * FROM count_values WHERE unique_values_count <= 1
    """
    params = {
        "model_type_id": MODEL_TYPE_ID,
        "non_tested_model_types": NON_TESTED_MODEL_TYPES
    }
    frozen_values_df = pd.read_sql(query, conn, params=params, index_col="id").sort_index()
    if frozen_values_df.empty:
        print("У всех моделей значения изменялись в течение последнего часа")
    else:
        print("У следующих моделей значения не изменяются за последний час:")
        print(frozen_values_df)

print("="*100)

У следующих моделей значения не изменяются за последний час:
                                     name  unique_values_count
id                                                            
3184                    G-43-107:DT_Tkk_n                    1
24010              KUPN:U400:Butane:SumC4                    1
24042                      iBF_GRS3:SumC4                    1
24052                      BF_GRS4:DNP+45                    1
24054                       BF_GRS4:SumC4                    1
24058              BF_GRS4:SumUnsaturated                    1
26609                   GFU2:Butane:SumC3                    1
26699                GFU2:Isobutane:SumC3                    1
26701    KUPN:U400:Propane:SumUnsaturated                    1
26703              KUPN:U400:Butane:SumC3                    1
26713  KUPN:U400:Isobutane:SumUnsaturated                    1


In [17]:
# Поиск моделей без даты добавления модели
print("="*100)

with psycopg2.connect(**DB_ARGS) as conn:
    query = """
        SELECT ut.id, 
            ut.name,
            (ulv.json_value->>'ecaluation_date')::TIMESTAMP WITH TIME ZONE AS ecaluation_date
        FROM dictionary.union_tags ut
        JOIN result.union_last_values ulv ON ut.id = ulv.tag_id
            AND ut.source_type_id = %(model_type_id)s
        WHERE NOT ((ut.attributes->>'type_model')::INTEGER = ANY(%(non_tested_model_types)s))
        AND (ulv.json_value->>'ecaluation_date')::TIMESTAMP WITH TIME ZONE IS NULL
        AND ut.name NOT LIKE %(layer1_filter)s
        AND ut.name NOT LIKE %(layer2_filter)s
        AND ut.name NOT LIKE %(layer3_filter)s
    """
    params = {
        "model_type_id": MODEL_TYPE_ID,
        "non_tested_model_types": NON_TESTED_MODEL_TYPES,
        "layer1_filter": LAYER1_FILTER,
        "layer2_filter": LAYER2_FILTER,
        "layer3_filter": LAYER3_FILTER
    }
    ecaluation_df = pd.read_sql(query, conn, params=params, index_col="id").sort_index()
    if ecaluation_df.empty:
        print("У всех моделей указана дата добавления")
    else:
        print("У следующих моделей не указана дата добавления:")
        print(ecaluation_df)

print("="*100)

У следующих моделей не указана дата добавления:
                                     name ecaluation_date
id                                                       
10656               StSmesh:569_Aromatics            None
10657                 StSmesh:569_Benzene            None
10659                     StSmesh:569_D15            None
10660                     StSmesh:569_DNP            None
10661                     StSmesh:569_EBP            None
10662                    StSmesh:569_I100            None
10663                    StSmesh:569_I150            None
10664                     StSmesh:569_I70            None
10665                     StSmesh:569_IBP            None
10667                     StSmesh:569_MON            None
10668                 StSmesh:569_Olefins            None
10669                     StSmesh:569_RON            None
10670               StSmesh:570_Aromatics            None
10671                 StSmesh:570_Benzene            None
10673                   

In [18]:
# Поиск моделей без метрик ИПК
print("="*100)

with psycopg2.connect(**DB_ARGS) as conn:
    query = """
        WITH metrics AS (
            SELECT ut.id,
                ut.name,
                ulv.json_value ? 'corr' AS corr,
                ulv.json_value ? 'det' AS det,
                ulv.json_value ? 'mae' AS mae,
                ulv.json_value ? 'Ir' AS Ir,
                ulv.json_value ? 'Icoef_det' AS Icoef_det,
                ulv.json_value ? 'Imae' AS Imae,
                ulv.json_value ? 'WI_corr' AS WI_corr,
                ulv.json_value ? 'WI_det' AS WI_det,
                ulv.json_value ? 'WI_mae' AS WI_mae,
                ulv.json_value ? 'ipk' AS ipk,
                ulv.json_value ? 'sko_la' AS sko_la
            FROM dictionary.union_tags ut
            JOIN result.union_last_values ulv ON ut.id = ulv.tag_id
                AND ut.source_type_id = %(model_type_id)s
            WHERE NOT ((ut.attributes->>'type_model')::INTEGER = ANY(%(non_tested_model_types)s))
            AND ut.name NOT LIKE %(layer1_filter)s
            AND ut.name NOT LIKE %(layer2_filter)s
            AND ut.name NOT LIKE %(layer3_filter)s
            AND ut.disabled = False
            AND (ut.attributes->>'view_diag')::BOOLEAN=TRUE
            AND ut.parent_id IS NOT NULL
        )
        SELECT * FROM metrics
        WHERE NOT corr OR NOT det OR NOT mae
            OR NOT Ir OR NOT Icoef_det OR NOT Imae
            OR NOT WI_corr OR NOT WI_det OR NOT WI_mae
            OR NOT ipk OR NOT sko_la
    """
    params = {
        "model_type_id": MODEL_TYPE_ID,
        "non_tested_model_types": NON_TESTED_MODEL_TYPES,
        "layer1_filter": LAYER1_FILTER,
        "layer2_filter": LAYER2_FILTER,
        "layer3_filter": LAYER3_FILTER
    }
    metrics_df = pd.read_sql(query, conn, params=params, index_col="id").sort_index()
    if metrics_df.empty:
        print("У всех моделей указаны метрики ИПК")
    else:
        print("У следующих моделей не указаны метрики ИПК (False - отсутствует метрика):")
        print(metrics_df)

print("="*100)

У следующих моделей не указаны метрики ИПК (False - отсутствует метрика):
                                       name   corr    det    mae     ir  icoef_det   imae  wi_corr  wi_det  wi_mae   ipk  sko_la
id                                                                                                                              
480                           GFU2:Benz:RON  False  False   True  False      False  False    False   False   False  True   False
481                           GFU2:Benz:MON   True   True   True   True       True   True    False   False   False  True    True
531                         GFU2:Butane:RON   True   True   True   True       True   True    False   False   False  True    True
581                          MTBE:MTBE:MTBE   True   True   True   True       True   True    False   False   False  True    True
614                         GFU2:Butane:MON   True   True   True   True       True   True    False   False   False  True    True
967                  GO

In [ ]:
# Проверка на соответствие бинарников и типов моделей


from core.model_core_sync import unpickle
import pickle

from app.models.mLinearRegression import mLinearRegression
from app.models.mSmesTemp import Smes_temp_model
from app.models.mSmesTempVols import Smes_temp_vols_model
from app.models.mFlashPoint import Smes_fp_model
from app.models.mDNPModel import DNP_model
from app.models.mCloudPoint import Smes_cp_model
from app.models.mCrystKerosene import CrystKerosene
from app.models.mRandomForest import mRandomForest
from app.models.mGradientBoosting import mGradientBoosting
from app.models.mPipeline import mPipeline
from app.models.mMixBonus import Mix_Bonus
from app.models.mMixDupont import Mix_Dupont
from app.models.mMixEthyl import Mix_Ethyl
from app.models.mSmesModel import Smes_model

from app.models.mReservQuality import GetReservQuality

from app.models.mOptimizerModel import OptimizerModel

model_binaries = {
    0: [mLinearRegression, Smes_temp_model, Smes_temp_vols_model,
        Smes_fp_model, DNP_model, Smes_cp_model,
        CrystKerosene, mRandomForest, mGradientBoosting,
        mPipeline, Mix_Bonus, Mix_Dupont,
        Mix_Ethyl, Smes_model, ],
    1: [Smes_model, ],
    # 5: ?
    # 6: ?
    # 7: ?
    10: [GetReservQuality, ],
    11: [OptimizerModel,],
}

# Функции
##############################################################################################################

def get_clusters(conn, model_id):
    query = """
        WITH clusters AS (
            SELECT id_model AS model_id,
                id AS cluster_id,
                id_cluster AS cluster,
                RANK() OVER(
                    PARTITION BY id_model, id_cluster 
                    ORDER BY date DESC
                ) AS cluster_order
            FROM model.model_cluster_config
            WHERE id_model = %(model_id)s
        ) 
        SELECT * FROM clusters
        WHERE cluster_order = 1
    """
    params = {"model_id": model_id}
    df = pd.read_sql(query, conn, params=params).set_index('cluster_id')
    return df

def get_binary(conn, model_id: int, cluster: int):
    query = """
        WITH binaries AS (
            SELECT bin_data,
                row_number() OVER (
                    PARTITION BY id_model
                    ORDER BY date desc
                ) binary_order
            FROM model.model_binary
            WHERE id_model = %(model_id)s
            AND cluster_id = %(cluster)s
        )
        SELECT bin_data FROM binaries
        WHERE binary_order = 1
        LIMIT 1
    """
    params = {"model_id": int(model_id), "cluster": int(cluster)}
    model = pd.read_sql(query, conn, params=params)
    binary = unpickle(model.at[0, "bin_data"])
    return binary


def get_models(conn):
    query = """
        SELECT id, name, (attributes->>'type_model')::INTEGER AS type_model
        FROM dictionary.union_tags
        WHERE source_type_id = %(model_type_id)s
        AND disabled = False
        AND (attributes->>'type_model')::INTEGER IS NOT NULL
        AND NOT ((attributes->>'type_model')::INTEGER = ANY(%(non_tested_model_types)s))
        AND name NOT LIKE %(layer1_filter)s
        AND name NOT LIKE %(layer2_filter)s
        AND name NOT LIKE %(layer3_filter)s
        AND (attributes->>'view_diag')::BOOLEAN=TRUE
    """
    params = {
        "model_type_id": MODEL_TYPE_ID,
        "non_tested_model_types": NON_TESTED_MODEL_TYPES,
        "layer1_filter": LAYER1_FILTER,
        "layer2_filter": LAYER2_FILTER,
        "layer3_filter": LAYER3_FILTER
    }
    df = pd.read_sql(query, conn, params=params, index_col="id").sort_index()
    return df

##############################################################################################################

print("="*100)

with psycopg2.connect(**DB_ARGS) as conn:
    models = get_models(conn)
    incorrect_binaries = []
    for model_id, model_row in models.iterrows():
        type_model = model_row["type_model"]
        name = model_row["name"]
        clusters_df = get_clusters(conn, model_id)
        for cluster_id, cluster_row in clusters_df.iterrows():
            cluster = cluster_row["cluster"]
            try:
                binary = get_binary(conn, model_id, cluster)
            except ModuleNotFoundError:
                incorrect_binaries.append({
                    "model_id": model_id, "name": name, "cluster": cluster,
                    "type_model": type_model, "binary_class": None, "no_binary": False,
                    "cannot_unpickle": True
                })
                continue
            except KeyError:
                incorrect_binaries.append({
                    "model_id": model_id, "name": name, "cluster": cluster,
                    "type_model": type_model, "binary_class": None, "no_binary": True,
                    "cannot_unpickle": False
                })
                continue
            except pickle.UnpicklingError:
                incorrect_binaries.append({
                    "model_id": model_id, "name": name, "cluster": cluster,
                    "type_model": type_model, "binary_class": None, "no_binary": False,
                    "cannot_unpickle": True
                })
                continue
    
            try:
                model_type_binaries = model_binaries[type_model]
            except KeyError:
                incorrect_binaries.append({
                    "model_id": model_id, "name": name, "cluster": cluster,
                    "type_model": type_model, "binary_class": type(binary).__name__, "no_binary": False,
                    "cannot_unpickle": False
                })
                continue
                
            is_correct_binary = False
            for mtb in model_type_binaries:
                if isinstance(binary, mtb):
                    is_correct_binary = True
            if not is_correct_binary:
                incorrect_binaries.append({
                    "model_id": model_id, "name": name, "cluster": cluster,
                    "type_model": type_model, "binary_class": type(binary).__name__, "no_binary": False,
                    "cannot_unpickle": False
                })
    incorrect_binaries_df = pd.DataFrame(incorrect_binaries)
    if incorrect_binaries_df.empty:
        print("У всех моделей соответстувют бинарники и типы моделей")
    else:
        incorrect_binaries_df = incorrect_binaries_df.set_index(["model_id", "cluster"]).sort_index()
        print("У следующих моделей не соответстувют бинарники и типы моделей:")
        print(incorrect_binaries_df)

print("="*100)

У следующих моделей не соответстувют бинарники и типы моделей:
                                name  type_model       binary_class  no_binary  cannot_unpickle
model_id cluster                                                                               
2760     0        GFU2:Butane:DNP+45           0  mLinearRegression      False            False
10825    0             TAME:315_I100           0               list      False            False
10826    0             TAME:316_I100           0               list      False            False


In [20]:
# Проверка предикторов, у которых нет значений за последний час
# Локально скрипт выполняется около 5-10 минут

NON_CONTINUOUS_SOURCE_TYPES = [2, 3, 16, 27]

# Функции (частично совпадают с предыдущим скриптом)
##############################################################################################################

def get_clusters(conn, model_id):
    query = """
        WITH clusters AS (
            SELECT id_model AS model_id,
                id AS cluster_id,
                id_cluster AS cluster,
                RANK() OVER(
                    PARTITION BY id_model, id_cluster 
                    ORDER BY date DESC
                ) AS cluster_order
            FROM model.model_cluster_config
            WHERE id_model = %(model_id)s
        ) 
        SELECT * FROM clusters
        WHERE cluster_order = 1
    """
    params = {"model_id": model_id}
    df = pd.read_sql(query, conn, params=params).set_index('cluster_id')
    return df


def get_models(conn):
    query = """
        SELECT id, name, (attributes->>'type_model')::INTEGER AS type_model
        FROM dictionary.union_tags
        WHERE source_type_id = %(model_type_id)s
        AND disabled = False
        AND (attributes->>'type_model')::INTEGER IS NOT NULL
        AND NOT ((attributes->>'type_model')::INTEGER = ANY(%(non_tested_model_types)s))
        AND name NOT LIKE %(layer1_filter)s
        AND name NOT LIKE %(layer2_filter)s
        AND name NOT LIKE %(layer3_filter)s
        AND (attributes->>'view_diag')::BOOLEAN=TRUE
    """
    params = {
        "model_type_id": MODEL_TYPE_ID,
        "non_tested_model_types": NON_TESTED_MODEL_TYPES,
        "layer1_filter": LAYER1_FILTER,
        "layer2_filter": LAYER2_FILTER,
        "layer3_filter": LAYER3_FILTER
    }
    df = pd.read_sql(query, conn, params=params, index_col="id").sort_index()
    return df


def get_predictor_last_values(conn, model_id, cluster_id):
    query = """
        SELECT mc.id_tag AS tag_id,
            mc.id_model AS model_id,
            model.name AS model_name, 
            ut.name AS tag_name, 
            ut.source_type_id, 
            ulv.date_value AS last_date,
            ulv.date_value < (NOW() - interval '1 hour') OR ulv.date_value IS NULL AS outdated
        FROM model.model_config mc
        JOIN dictionary.union_tags ut ON mc.id_tag = ut.id
        JOIN dictionary.union_tags model ON mc.id_model = model.id
        LEFT JOIN result.union_last_values ulv ON mc.id_tag = ulv.tag_id
        WHERE mc.id_model = %(model_id)s
            AND mc.id_cluster = %(cluster_id)s
            AND NOT ut.source_type_id = ANY(%(non_continuous_types)s)
    """
    params = {
        "model_id": model_id,
        "cluster_id": cluster_id,
        "non_continuous_types": NON_CONTINUOUS_SOURCE_TYPES
    }
    df = pd.read_sql(query, conn, params=params).groupby(["tag_id", "model_id"]).first()
    return df

##############################################################################################################

print("="*100)

with psycopg2.connect(**DB_ARGS) as conn:
    models = get_models(conn)
    outdated_preds = pd.DataFrame(
        columns=["tag_id", "model_id", "model_name", "tag_name", 
                 "source_type_id", "last_date", "outdated"]
    ).set_index(["tag_id", "model_id"])
    for model_id, model_row in models.iterrows():
        type_model = model_row["type_model"]
        name = model_row["name"]
        clusters_df = get_clusters(conn, model_id)
        for cluster_id, cluster_row in clusters_df.iterrows():
            cluster = cluster_row["cluster"]
            pred_lv = get_predictor_last_values(conn, model_id, cluster_id)
            outdated_pred_lv = pred_lv[pred_lv["outdated"]]
            outdated_preds = pd.concat([outdated_preds, outdated_pred_lv])
    outdated_preds = outdated_preds.sort_index()
    if outdated_preds.empty:
        print("У всех предикторов есть значения за последний час")
    else:
        outdated_preds = outdated_preds.sort_index()
        print("У следующих предикторов нет значений за последний час:")
        print(outdated_preds)

print("="*100)

У следующих предикторов нет значений за последний час:
                model_name            tag_name source_type_id                 last_date outdated
tag_id model_id                                                                                 
1142   10650     P-577:T50  35-11-1000:RIF:T50              6 2024-10-08 05:49:00+00:00     True
       10651     P-577:Tkk  35-11-1000:RIF:T50              6 2024-10-08 05:49:00+00:00     True
       10652     P-577:Tnk  35-11-1000:RIF:T50              6 2024-10-08 05:49:00+00:00     True
1144   10650     P-577:T50  35-11-1000:RIF:IBP              6 2024-10-08 05:49:00+00:00     True
       10651     P-577:Tkk  35-11-1000:RIF:IBP              6 2024-10-08 05:49:00+00:00     True
       10652     P-577:Tnk  35-11-1000:RIF:IBP              6 2024-10-08 05:49:00+00:00     True
1145   10650     P-577:T50  35-11-1000:RIF:EBP              6 2024-10-08 05:49:00+00:00     True
       10651     P-577:Tkk  35-11-1000:RIF:EBP              6 2024-10-08

In [22]:
# Проверка количества предикторов в бинарнике и конфиге
# Локально скрипт выполняется около 5-10 минут

from core.model_core_sync import unpickle
import numpy as np
import pickle
from app.models.mLinearRegression import mLinearRegression

LINEAR_TYPE_MODEL = 0

# Функции (частично совпадают с предыдущими скриптами)
############################################################################################################## 

def get_clusters(conn, model_id):
    query = """
        WITH clusters AS (
            SELECT id_model AS model_id,
                id AS cluster_id,
                id_cluster AS cluster,
                RANK() OVER(
                    PARTITION BY id_model, id_cluster 
                    ORDER BY date DESC
                ) AS cluster_order
            FROM model.model_cluster_config
            WHERE id_model = %(model_id)s
        ) 
        SELECT * FROM clusters
        WHERE cluster_order = 1
    """
    params = {"model_id": model_id}
    df = pd.read_sql(query, conn, params=params).set_index('cluster_id')
    return df

def get_binary(conn, model_id: int, cluster: int):
    try:
        query = """
            WITH binaries AS (
                SELECT bin_data,
                    row_number() OVER (
                        PARTITION BY id_model
                        ORDER BY date desc
                    ) binary_order
                FROM model.model_binary
                WHERE id_model = %(model_id)s
                AND cluster_id = %(cluster)s
            )
            SELECT bin_data FROM binaries
            WHERE binary_order = 1
            LIMIT 1
        """
        params = {"model_id": int(model_id), "cluster": int(cluster)}
        model = pd.read_sql(query, conn, params=params)
        binary = unpickle(model.at[0, "bin_data"])
        return binary
    except: (ModuleNotFoundError, pickle.UnpicklingError)

def get_linear_models(conn):
    query = """
        SELECT id, name
        FROM dictionary.union_tags
        WHERE source_type_id = %(model_type_id)s
        AND disabled = False
        AND (attributes->>'type_model')::INTEGER = %(linear_type_model)s
        AND (attributes->>'view_diag')::BOOLEAN=TRUE
    """
    params = {
        "model_type_id": MODEL_TYPE_ID, 
        "linear_type_model": LINEAR_TYPE_MODEL
    }
    df = pd.read_sql(query, conn, params=params, index_col="id").sort_index()
    return df

def get_predictor_count(conn, model_id, cluster_id):
    query = """
        SELECT COUNT(*) AS cnt
        FROM model.model_config
        WHERE id_model = %(model_id)s
            AND id_cluster = %(cluster_id)s
    """
    params = {
        "model_id": model_id,
        "cluster_id": cluster_id,
    }
    df = pd.read_sql(query, conn, params=params)
    return 0 if df.empty else df.at[0, "cnt"]

##############################################################################################################

print("="*100)

with psycopg2.connect(**DB_ARGS) as conn:
        linear_models = get_linear_models(conn)
        diff_pred_count = []
        for model_id, model_row in linear_models.iterrows():
            name = model_row["name"]
            clusters_df = get_clusters(conn, model_id)
            for cluster_id, cluster_row in clusters_df.iterrows():
                cluster = cluster_row["cluster"]

                try:
                    binary = get_binary(conn, model_id, cluster)
                except (ModuleNotFoundError, pickle.UnpicklingError):
                    continue
                    
                if isinstance(binary, mLinearRegression):
                    config_pred_count = get_predictor_count(conn, model_id, cluster_id)
                    binary_pred_count = len(np.ravel(binary.coef_))
                    if config_pred_count != binary_pred_count:
                        diff_pred_count.append({
                            "model_id": model_id,
                            "cluster": cluster,
                            "name": name,
                            "config_pred_count": config_pred_count,
                            "binary_pred_count": binary_pred_count
                        })
        diff_pred_count_df = pd.DataFrame(diff_pred_count)
        if diff_pred_count_df.empty:
            print("У всех линейных моделей совпадает число предикторов в бинарнике и конфигурации")
        else:
            diff_pred_count_df = diff_pred_count_df.set_index(["model_id", "cluster"]).sort_index()
            print("У следующих линейных моделей не совпадает число предикторов в бинарнике и конфигурации")
            print(diff_pred_count_df)

print("="*100)

У следующих линейных моделей не совпадает число предикторов в бинарнике и конфигурации
                                name  config_pred_count  binary_pred_count
model_id cluster                                                          
2760     1        GFU2:Butane:DNP+45                  5                  3
